# MMD Instruction Training — Kaggle GPU
Launch manually in Kaggle and attach one prepared MCT instruction dataset containing train.jsonl, validation.jsonl, test.jsonl and provenance.json. This trains from random initialization; no pretrained weights are loaded.

In [ ]:
!git clone --depth 1 --branch feat/mmd-cloud-training-mct-02 https://github.com/pctaha501-create/MMD-GameForge.git /kaggle/working/MMD-GameForge
%pip -q install tokenizers
import pathlib, torch
REPO=pathlib.Path('/kaggle/working/MMD-GameForge')
matches=list(pathlib.Path('/kaggle/input').rglob('provenance.json'))
if len(matches)!=1: raise RuntimeError(f'Expected exactly one instruction provenance.json; found {len(matches)}')
DATA_DIR=matches[0].parent
for name in ('train.jsonl','validation.jsonl','test.jsonl'):
    if not (DATA_DIR/name).exists(): raise FileNotFoundError(DATA_DIR/name)
OUTPUT=pathlib.Path('/kaggle/working/mmd-instruction-run')
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'not available')
!python /kaggle/working/MMD-GameForge/cloud/runtime/model_factory.py create --profile multilingual-tiny --output /kaggle/working/mmd-instruction-config.json


In [ ]:
!python /kaggle/working/MMD-GameForge/cloud/runtime/train_instructions.py --config /kaggle/working/mmd-instruction-config.json --dataset-dir "$DATA_DIR" --output "$OUTPUT"


In [ ]:
!python /kaggle/working/MMD-GameForge/cloud/runtime/bundle.py --checkpoint-dir "$OUTPUT" --config /kaggle/working/mmd-instruction-config.json --evaluation "$OUTPUT/instruction-evaluation.json" --output /kaggle/working/MMD-instruction-model.zip
print('Exported: /kaggle/working/MMD-instruction-model.zip')
